In [1]:
import pandas as pd
import os

# 🔴 PASTE YOUR FOLDER PATH HERE
folder_path = "/Users/nagesh/Documents/archive"

all_files = os.listdir(folder_path)

df_list = []

for file in all_files:
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

print("Total rows:", combined_df.shape)
combined_df.head()
combined_df.columns

/var/folders/m3/jkhk51dj5yq7rl4p7yyq52sr0000gn/T/ipykernel_1452/3752004877.py:14: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Total rows: (3434754, 21)


Index(['StationId', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO',
       'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket',
       'City', 'Datetime', 'StationName', 'State', 'Status'],
      dtype='object')

In [2]:
# Remove rows where AQI is missing
combined_df = combined_df.dropna(subset=['AQI'])

# Convert Date to datetime
combined_df['Date'] = pd.to_datetime(combined_df['Date'])

# Sort values
combined_df = combined_df.sort_values(['City', 'Date'])

print("Cleaned dataset shape:", combined_df.shape)

Cleaned dataset shape: (2709563, 21)


In [3]:
# Check available cities
combined_df['City'].unique()[:10]

array(['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru',
       'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore'],
      dtype=object)

In [4]:
city_name = "Delhi"   # change if needed

city_df = combined_df[combined_df['City'] == city_name]

print(city_df.shape)
city_df.head()

(49693, 21)


,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,Benzene,Toluene,Xylene,AQI,AQI_Bucket,City,Datetime,StationName,State,Status
826139,NaN,2015-01-01,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,...,14.36,24.86,9.84,472.0,Severe,Delhi,NaN,NaN,NaN,NaN
826140,NaN,2015-01-02,186.18,269.55,62.09,32.87,88.14,31.83,9.54,6.65,...,10.55,20.09,4.29,454.0,Severe,Delhi,NaN,NaN,NaN,NaN
826141,NaN,2015-01-03,87.18,131.90,25.73,30.31,47.95,69.55,10.61,2.65,...,3.91,10.23,1.99,143.0,Moderate,Delhi,NaN,NaN,NaN,NaN
826142,NaN,2015-01-04,151.84,241.84,25.01,36.91,48.62,130.36,11.54,4.63,...,4.26,9.71,3.34,319.0,Very Poor,Delhi,NaN,NaN,NaN,NaN
826143,NaN,2015-01-05,146.60,219.13,14.01,34.92,38.25,122.88,9.20,3.33,...,2.80,6.21,2.96,325.0,Very Poor,Delhi,NaN,NaN,NaN,NaN


In [5]:
features = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
target = 'AQI'

# Drop rows with missing feature values
city_df = city_df.dropna(subset=features)

X = city_df[features]
y = city_df[target]

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))

MAE: 39.34260668103448


In [7]:
import joblib

joblib.dump(model, "aqi_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [8]:
city_name = "Delhi"
city_df = combined_df[combined_df['City'] == city_name]

In [9]:
city_df = city_df[['Date', 'AQI']]
city_df = city_df.dropna()

city_df = city_df.groupby('Date').mean().reset_index()

city_df = city_df.sort_values('Date')
city_df.set_index('Date', inplace=True)

city_df.head()

,AQI
Date,
2015-01-01,472.0
2015-01-02,454.0
2015-01-03,143.0
2015-01-04,319.0
2015-01-05,325.0


In [24]:
pip install prophet

In [ ]:
from prophet import Prophet

# Prepare data for Prophet
city_df = combined_df[combined_df['City'] == "Delhi"]

city_df = city_df[['Date', 'AQI']]
city_df = city_df.dropna()

city_df = city_df.groupby('Date').mean().reset_index()

forecast_df = city_df.rename(columns={"Date": "ds", "AQI": "y"})

# Train Prophet model
model_prophet = Prophet()
model_prophet.fit(forecast_df)

print("Forecast model trained successfully!")

14:49:15 - cmdstanpy - INFO - Chain [1] start processing
14:49:15 - cmdstanpy - INFO - Chain [1] done processing


Forecast model trained successfully!


In [ ]:
future = model_prophet.make_future_dataframe(periods=7)
forecast = model_prophet.predict(future)

forecast[['ds', 'yhat']].tail(7)

,ds,yhat
1999,2020-07-02,65.482257
2000,2020-07-03,65.152883
2001,2020-07-04,57.281321
2002,2020-07-05,50.051064
2003,2020-07-06,44.923927
2004,2020-07-07,45.672084
2005,2020-07-08,48.072449


In [ ]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [ ]:
import joblib
joblib.dump(model_prophet, "forecast_model.pkl")
print("Forecast model saved!")

Forecast model saved!
